In [1]:
import os
import glob
import pandas as pd
import numpy as np
import random
from tqdm.notebook import tqdm
from IPython.display import display


def seed_it_all(seed=7):
    """Set reproducible seeds without importing TensorFlow."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    random.seed(seed)


seed_it_all()




In [2]:
print("\n... BASIC DATA SETUP STARTING ...\n")

DATA_DIR = "/kaggle/input/uw-madison-gi-tract-image-segmentation"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
train_df = pd.read_csv(TRAIN_CSV)

all_train_images = glob.glob(os.path.join(TRAIN_DIR, "**", "*.png"), recursive=True)

TEST_DIR = os.path.join(DATA_DIR, "test")
SS_CSV = os.path.join(DATA_DIR, "sample_submission.csv")
ss_df = pd.read_csv(SS_CSV)

all_test_images = glob.glob(os.path.join(TEST_DIR, "**", "*.png"), recursive=True)

DEBUG = len(ss_df) == 0
if DEBUG:
    TEST_DIR = TRAIN_DIR
    all_test_images = all_train_images
    ss_df = train_df.iloc[:10][["id", "class"]].copy()
    ss_df["predicted"] = ""

print(f"\n... ARE WE DEBUGGING: {DEBUG} ...\n")
print("\n... BASIC DATA SETUP FINISHED ...\n")





... BASIC DATA SETUP STARTING ...


... ARE WE DEBUGGING: False ...


... BASIC DATA SETUP FINISHED ...



In [3]:
def df_preprocessing(df, globbed_file_list, is_test=False):
    """Add useful columns and merge file paths."""
    df["case_id_str"] = df["id"].apply(lambda x: x.split("_", 2)[0])
    df["case_id"] = df["id"].apply(
        lambda x: int(x.split("_", 2)[0].replace("case", ""))
    )
    df["day_num_str"] = df["id"].apply(lambda x: x.split("_", 2)[1])
    df["day_num"] = df["id"].apply(lambda x: int(x.split("_", 2)[1].replace("day", "")))
    df["slice_id"] = df["id"].apply(lambda x: x.split("_", 2)[2])

    base_path = globbed_file_list[0].rsplit("/", 4)[0] + "/"
    df["_partial_ident"] = (
        base_path
        + df["case_id_str"]
        + "/"
        + df["case_id_str"]
        + "_"
        + df["day_num_str"]
        + "/scans/"
        + df["slice_id"]
    )

    tmp = pd.DataFrame(
        {
            "_partial_ident": [x.rsplit("_", 4)[0] for x in globbed_file_list],
            "f_path": globbed_file_list,
        }
    )
    df = df.merge(tmp, on="_partial_ident").drop(columns=["_partial_ident"])

    df["slice_h"] = df["f_path"].apply(lambda x: int(x[:-4].rsplit("_", 4)[1]))
    df["slice_w"] = df["f_path"].apply(lambda x: int(x[:-4].rsplit("_", 4)[2]))
    df["px_spacing_h"] = df["f_path"].apply(lambda x: float(x[:-4].rsplit("_", 4)[3]))
    df["px_spacing_w"] = df["f_path"].apply(lambda x: float(x[:-4].rsplit("_", 4)[4]))

    if not is_test:
        l_bowel = df[df["class"] == "large_bowel"][["id", "segmentation"]].rename(
            columns={"segmentation": "lb_seg_rle"}
        )
        s_bowel = df[df["class"] == "small_bowel"][["id", "segmentation"]].rename(
            columns={"segmentation": "sb_seg_rle"}
        )
        stomach = df[df["class"] == "stomach"][["id", "segmentation"]].rename(
            columns={"segmentation": "st_seg_rle"}
        )
        df = df.merge(l_bowel, on="id", how="left")
        df = df.merge(s_bowel, on="id", how="left")
        df = df.merge(stomach, on="id", how="left")
        df = df.drop_duplicates(subset=["id"]).reset_index(drop=True)
        df["lb_seg_flag"] = df["lb_seg_rle"].apply(lambda x: not pd.isna(x))
        df["sb_seg_flag"] = df["sb_seg_rle"].apply(lambda x: not pd.isna(x))
        df["st_seg_flag"] = df["st_seg_rle"].apply(lambda x: not pd.isna(x))
        df["n_segs"] = (
            df["lb_seg_flag"].astype(int)
            + df["sb_seg_flag"].astype(int)
            + df["st_seg_flag"].astype(int)
        )

    new_order = [
        "id",
        "f_path",
        "n_segs",
        "lb_seg_rle",
        "lb_seg_flag",
        "sb_seg_rle",
        "sb_seg_flag",
        "st_seg_rle",
        "st_seg_flag",
        "slice_h",
        "slice_w",
        "px_spacing_h",
        "px_spacing_w",
        "case_id_str",
        "case_id",
        "day_num_str",
        "day_num",
        "slice_id",
    ]
    if is_test:
        new_order.insert(1, "class")
    new_order = [c for c in new_order if c in df.columns]
    df = df[new_order]
    return df




In [4]:
train_df = df_preprocessing(train_df, all_train_images, is_test=False)
ss_df = df_preprocessing(ss_df, all_test_images, is_test=True)

print("Training dataframe sample:")
display(train_df.head())
print("\nSubmission dataframe sample (pre‑prediction):")
display(ss_df.head())




Training dataframe sample:


,id,f_path,n_segs,lb_seg_rle,lb_seg_flag,sb_seg_rle,sb_seg_flag,st_seg_rle,st_seg_flag,slice_h,slice_w,px_spacing_h,px_spacing_w,case_id_str,case_id,day_num_str,day_num,slice_id
0,case77_day20_slice_0001,/kaggle/input/uw-madison-gi-tract-image-segmen...,0,NaN,False,NaN,False,NaN,False,266,266,1.5,1.5,case77,77,day20,20,slice_0001
1,case77_day20_slice_0002,/kaggle/input/uw-madison-gi-tract-image-segmen...,0,NaN,False,NaN,False,NaN,False,266,266,1.5,1.5,case77,77,day20,20,slice_0002
2,case77_day20_slice_0003,/kaggle/input/uw-madison-gi-tract-image-segmen...,0,NaN,False,NaN,False,NaN,False,266,266,1.5,1.5,case77,77,day20,20,slice_0003
3,case77_day20_slice_0004,/kaggle/input/uw-madison-gi-tract-image-segmen...,0,NaN,False,NaN,False,NaN,False,266,266,1.5,1.5,case77,77,day20,20,slice_0004
4,case77_day20_slice_0005,/kaggle/input/uw-madison-gi-tract-image-segmen...,0,NaN,False,NaN,False,NaN,False,266,266,1.5,1.5,case77,77,day20,20,slice_0005



Submission dataframe sample (pre‑prediction):


,id,class,f_path,slice_h,slice_w,px_spacing_h,px_spacing_w,case_id_str,case_id,day_num_str,day_num,slice_id
0,case123_day20_slice_0001,large_bowel,/kaggle/input/uw-madison-gi-tract-image-segmen...,266,266,1.5,1.5,case123,123,day20,20,slice_0001
1,case123_day20_slice_0001,small_bowel,/kaggle/input/uw-madison-gi-tract-image-segmen...,266,266,1.5,1.5,case123,123,day20,20,slice_0001
2,case123_day20_slice_0001,stomach,/kaggle/input/uw-madison-gi-tract-image-segmen...,266,266,1.5,1.5,case123,123,day20,20,slice_0001
3,case123_day20_slice_0002,large_bowel,/kaggle/input/uw-madison-gi-tract-image-segmen...,266,266,1.5,1.5,case123,123,day20,20,slice_0002
4,case123_day20_slice_0002,small_bowel,/kaggle/input/uw-madison-gi-tract-image-segmen...,266,266,1.5,1.5,case123,123,day20,20,slice_0002


In [5]:
slice_px_map = {}
cols = ["slice_h", "slice_w", "px_spacing_h", "px_spacing_w"]
for _, row in (
    train_df[train_df["n_segs"] == 3]
    .groupby(cols)[["lb_seg_rle", "sb_seg_rle", "st_seg_rle"]]
    .first()
    .reset_index()
    .iterrows()
):
    key_base = (
        f"{row['slice_h']}-{row['slice_w']}-{row['px_spacing_h']}-{row['px_spacing_w']}"
    )
    slice_px_map[f"{key_base}-large_bowel"] = row["lb_seg_rle"]
    slice_px_map[f"{key_base}-small_bowel"] = row["sb_seg_rle"]
    slice_px_map[f"{key_base}-stomach"] = row["st_seg_rle"]

print(f"Created lookup for {len(slice_px_map)} slice‑class combinations.")




Created lookup for 9 slice‑class combinations.


In [6]:
ss_df["sli"] = ss_df["slice_id"].apply(lambda x: int(x.split("_")[-1]))
display(ss_df.head())




,id,class,f_path,slice_h,slice_w,px_spacing_h,px_spacing_w,case_id_str,case_id,day_num_str,day_num,slice_id,sli
0,case123_day20_slice_0001,large_bowel,/kaggle/input/uw-madison-gi-tract-image-segmen...,266,266,1.5,1.5,case123,123,day20,20,slice_0001,1
1,case123_day20_slice_0001,small_bowel,/kaggle/input/uw-madison-gi-tract-image-segmen...,266,266,1.5,1.5,case123,123,day20,20,slice_0001,1
2,case123_day20_slice_0001,stomach,/kaggle/input/uw-madison-gi-tract-image-segmen...,266,266,1.5,1.5,case123,123,day20,20,slice_0001,1
3,case123_day20_slice_0002,large_bowel,/kaggle/input/uw-madison-gi-tract-image-segmen...,266,266,1.5,1.5,case123,123,day20,20,slice_0002,2
4,case123_day20_slice_0002,small_bowel,/kaggle/input/uw-madison-gi-tract-image-segmen...,266,266,1.5,1.5,case123,123,day20,20,slice_0002,2


In [7]:
ss_df["ident"] = (
    ss_df["slice_h"].astype(str)
    + "-"
    + ss_df["slice_w"].astype(str)
    + "-"
    + ss_df["px_spacing_h"].astype(str)
    + "-"
    + ss_df["px_spacing_w"].astype(str)
    + "-"
    + ss_df["class"]
)
ss_df["predicted"] = ss_df["ident"].map(slice_px_map)




In [8]:
# Further restrict predictions: keep only stomach masks on slice 80
ss_df["predicted"] = ss_df.apply(
    lambda x: (
        x["predicted"]
        if (80 <= x["sli"] <= 80) and (x["class"] == "stomach")
        else np.nan
    ),
    axis=1,
)




In [9]:
submission = ss_df[["id", "class", "predicted"]]
submission.to_csv("submission.csv", index=False)
print("Submission file written to submission.csv")
display(submission.head())

Submission file written to submission.csv


,id,class,predicted
0,case123_day20_slice_0001,large_bowel,NaN
1,case123_day20_slice_0001,small_bowel,NaN
2,case123_day20_slice_0001,stomach,NaN
3,case123_day20_slice_0002,large_bowel,NaN
4,case123_day20_slice_0002,small_bowel,NaN
